# 03 - Portfolio Targets

Use this notebook after generating momentum rankings. It walks down the ranked list and suggests buy quantities using the ATR-based formula:

`shares = account_value * daily_move_target / atr_20`

The default behavior only buys a candidate when there is enough cash for the full suggested share count.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from clenow.portfolio import build_buy_list

In [2]:
# Portfolio inputs. Edit these before each weekly run.
ACCOUNT_VALUE = 100_000.00
AVAILABLE_CASH = 100_000.00
DAILY_MOVE_TARGET = 0.003  # 10 basis points of account value per ATR20 move
ALLOW_PARTIAL_FINAL_POSITION = False

# Optional: add current holdings if you want shares_to_buy to be a delta to target.
# Example: {"AAPL": 25, "MSFT": 10}
EXISTING_POSITIONS = {}

In [3]:
RANKINGS_PATH = PROJECT_ROOT / "data" / "processed" / "momentum_rankings.parquet"
BUY_LIST_PATH = PROJECT_ROOT / "data" / "processed" / "buy_list.csv"
DIAGNOSTICS_PATH = PROJECT_ROOT / "data" / "processed" / "buy_list_diagnostics.csv"

rankings = pd.read_parquet(RANKINGS_PATH)
print(f"Loaded {len(rankings)} ranked candidates")
rankings.head()

Loaded 205 ranked candidates


,rank,ticker,date,last_price,ATR20,MA100,avg_dollar_volume_20,candle_gap,max_gap_90,annualized_slope,r_squared,momentum_score,above_trend_ma,history_days
0,1,SNDK,2026-05-22,1478.689941,116.352704,763.578999,2.034255e+10,0.028420,0.130117,24.187924,0.861570,20.839589,True,320
1,2,CIEN,2026-05-22,583.739990,35.836505,382.623100,1.083841e+09,0.026142,0.126027,14.889722,0.956468,14.241541,True,753
2,3,LITE,2026-05-22,946.900024,84.689005,674.612800,5.872106e+09,0.019273,0.143062,15.909667,0.835655,13.294998,True,753
3,4,DELL,2026-05-22,295.190002,15.020998,162.690373,1.523937e+09,0.091771,0.103164,11.864750,0.950771,11.280654,True,753
4,5,INTC,2026-05-22,119.839996,8.884001,61.140400,1.669334e+10,0.004172,0.104947,15.171299,0.656575,9.961096,True,753


In [4]:
buy_list, diagnostics = build_buy_list(
    rankings,
    account_value=ACCOUNT_VALUE,
    available_cash=AVAILABLE_CASH,
    daily_move_target=DAILY_MOVE_TARGET,
    atr_column="ATR20",
    existing_positions=EXISTING_POSITIONS,
    allow_partial_final_position=ALLOW_PARTIAL_FINAL_POSITION,
)

buy_list.to_csv(BUY_LIST_PATH, index=False)
diagnostics.to_csv(DIAGNOSTICS_PATH, index=False)

estimated_total = buy_list["estimated_cost"].sum() if not buy_list.empty else 0.0
remaining_cash = AVAILABLE_CASH - estimated_total
print(f"Suggested {len(buy_list)} buys")
print(f"Estimated spend: ${estimated_total:,.2f}")
print(f"Remaining cash: ${remaining_cash:,.2f}")
buy_list.head(50)

Suggested 20 buys
Estimated spend: $99,979.73
Remaining cash: $20.27


,rank,ticker,date,last_price,ATR20,MA100,avg_dollar_volume_20,candle_gap,max_gap_90,annualized_slope,r_squared,momentum_score,above_trend_ma,history_days,target_shares,shares_to_buy,estimated_cost,remaining_cash
0,1,SNDK,2026-05-22,1478.689941,116.352704,763.578999,2.034255e+10,0.028420,0.130117,24.187924,0.861570,20.839589,True,320,2,2,2957.379883,97042.620117
1,2,CIEN,2026-05-22,583.739990,35.836505,382.623100,1.083841e+09,0.026142,0.126027,14.889722,0.956468,14.241541,True,753,8,8,4669.919922,92372.700195
2,3,LITE,2026-05-22,946.900024,84.689005,674.612800,5.872106e+09,0.019273,0.143062,15.909667,0.835655,13.294998,True,753,3,3,2840.700073,89532.000122
3,4,DELL,2026-05-22,295.190002,15.020998,162.690373,1.523937e+09,0.091771,0.103164,11.864750,0.950771,11.280654,True,753,19,19,5608.610046,83923.390076
4,5,INTC,2026-05-22,119.839996,8.884001,61.140400,1.669334e+10,0.004172,0.104947,15.171299,0.656575,9.961096,True,753,33,33,3954.719879,79968.670197
5,6,STX,2026-05-22,812.729980,49.380496,475.389870,3.529277e+09,0.002449,0.107782,9.336687,0.751424,7.015808,True,753,6,6,4876.379883,75092.290314
6,7,WDC,2026-05-22,484.279999,30.912003,314.856968,3.662958e+09,0.008569,0.122224,7.118080,0.831811,5.920896,True,753,9,9,4358.519989,70733.770325
7,8,VRT,2026-05-22,327.459991,18.900700,256.429684,1.930811e+09,0.019850,0.089969,6.095813,0.891507,5.434463,True,753,15,15,4911.899872,65821.870453
8,9,COHR,2026-05-22,377.570007,27.213503,265.652301,2.305883e+09,0.011733,0.143997,5.225678,0.863899,4.514457,True,753,11,11,4153.270081,61668.600372
9,10,GLW,2026-05-22,194.050003,13.788998,139.095191,2.875679e+09,0.001546,0.095216,5.252102,0.812214,4.265832,True,753,21,21,4075.050064,57593.550308


In [5]:
display_columns = [
    "rank",
    "ticker",
    "last_price",
    "ATR20",
    "momentum_score",
    "target_shares",
    "shares_to_buy",
    "estimated_cost",
    "remaining_cash",
]
buy_list[display_columns] if not buy_list.empty else buy_list

,rank,ticker,last_price,ATR20,momentum_score,target_shares,shares_to_buy,estimated_cost,remaining_cash
0,1,SNDK,1478.689941,116.352704,20.839589,2,2,2957.379883,97042.620117
1,2,CIEN,583.739990,35.836505,14.241541,8,8,4669.919922,92372.700195
2,3,LITE,946.900024,84.689005,13.294998,3,3,2840.700073,89532.000122
3,4,DELL,295.190002,15.020998,11.280654,19,19,5608.610046,83923.390076
4,5,INTC,119.839996,8.884001,9.961096,33,33,3954.719879,79968.670197
5,6,STX,812.729980,49.380496,7.015808,6,6,4876.379883,75092.290314
6,7,WDC,484.279999,30.912003,5.920896,9,9,4358.519989,70733.770325
7,8,VRT,327.459991,18.900700,5.434463,15,15,4911.899872,65821.870453
8,9,COHR,377.570007,27.213503,4.514457,11,11,4153.270081,61668.600372
9,10,GLW,194.050003,13.788998,4.265832,21,21,4075.050064,57593.550308


In [6]:
diagnostics.head(100)

,ticker,rank,target_shares,shares_to_buy,estimated_cost,remaining_cash,action
0,SNDK,1,2,2,2957.379883,97042.620117,buy
1,CIEN,2,8,8,4669.919922,92372.700195,buy
2,LITE,3,3,3,2840.700073,89532.000122,buy
3,DELL,4,19,19,5608.610046,83923.390076,buy
4,INTC,5,33,33,3954.719879,79968.670197,buy
...,...,...,...,...,...,...,...
95,LNT,96,238,238,17600.099274,20.270844,insufficient_cash
96,HWM,97,34,34,8722.699585,20.270844,insufficient_cash
97,NI,98,342,342,16364.699478,20.270844,insufficient_cash
98,SBAC,99,55,55,11306.350403,20.270844,insufficient_cash
